# Bench workshop app, from Colab

This notebook starts the Bench app on this Colab machine and gives you a link to open it in a new browser tab.
You don't need to read or edit any code. Run the four cells in order.

**Before you start:** open the **Secrets** panel (key icon on the left).
- `ARIZE_API_KEY` and `ARIZE_SPACE_ID` should already be there: you added them in the pre-read. Switch on notebook access for both, since that is set per notebook.
- Add two new ones, using the values your team shared with you internally: `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_API_KEY`. Switch on notebook access for these too.

## 1. Install

In [ ]:
!pip install -q uv
!uv pip install -q --system openai==2.31.0 openai-agents==0.13.6 arize==8.64.0 arize-otel==0.12.0 openinference-instrumentation-openai-agents==1.4.1 pandas==2.3.3 python-dotenv==1.2.2 streamlit==1.67.0
!git clone -q https://github.com/levelup-labs-ai/egen-workshop.git
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print("Installed.")

## 2. Check your keys

This makes one small call with each key, so a typo shows up here and not halfway through the session.

In [ ]:
from google.colab import userdata
from arize import ArizeClient
from openai import OpenAI

SECRETS = ["ARIZE_API_KEY", "ARIZE_SPACE_ID", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY"]

def secret(name):
    try:
        return userdata.get(name)
    except Exception:  # not added yet, or notebook access is switched off
        return None

keys = {name: secret(name) for name in SECRETS}
missing = [name for name, value in keys.items() if not value]
if missing:
    raise RuntimeError(f"Missing: {missing}. Add them in the Secrets panel (key icon) and switch on notebook access.")

model = OpenAI(base_url=keys["AZURE_OPENAI_ENDPOINT"].rstrip("/"), api_key=keys["AZURE_OPENAI_API_KEY"])
question = "Settle this in one sentence, and you have to pick one: which is the better city, San Francisco or Chicago?"
reply = model.chat.completions.create(model="gpt-5.4-mini", messages=[{"role": "user", "content": question}])
print("Model endpoint and key work. We asked it to settle an old LevelUp Labs argument:\n")
print("  ", reply.choices[0].message.content)

datasets = ArizeClient(api_key=keys["ARIZE_API_KEY"]).datasets.list(space=keys["ARIZE_SPACE_ID"]).datasets
print(f"\nArize key and space id work. Your space has {len(datasets)} dataset(s).")

## 3. Start the app

In [ ]:
import os, subprocess, time, urllib.request
from google.colab import userdata

# The app runs as its own program, so it can't read the Secrets panel. Hand it the secrets as environment variables.
SECRETS = ["ARIZE_API_KEY", "ARIZE_SPACE_ID", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY"]
app_env = os.environ | {name: userdata.get(name) for name in SECRETS}

app = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    cwd="egen-workshop", env=app_env, stdout=open("app.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):  # wait until the app answers
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health")
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("The app didn't start. Look at app.log.")
print("The app is running on this Colab machine.")

## 4. Get your link

This opens a tunnel: a temporary public web address that forwards to the app on this Colab machine.
The address is yours for this session. Don't share it: anyone who has it can use your keys through the app.

In [ ]:
import re

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:  # cloudflared prints the address once the tunnel is up
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        print("Open the app here:\n\n   ", found.group(0))
        break
else:
    raise RuntimeError("The tunnel didn't open. Your network may block trycloudflare.com.")

When you're done, stop the app and close the tunnel: **Runtime → Disconnect and delete runtime**.